# 05. 청년 이동패턴 검증

분석 흐름은 다음과 같습니다.

1. 20~39세 청년 이동자료의 품질 확인
2. 평일 오전 07:00~09:40 이동 집계
3. 평일 오후 17:00~19:40 이동의 방향을 반대로 변환하여 집계
4. 전체연령 출근·귀가 목적 OD와 청년 이동 OD 비교
5. 네 가지 비교 지표 산출

## 1. 라이브러리와 파일 경로 설정

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display


BASE_DIR = Path(
    r"C:\Users\Owner\OneDrive\바탕 화면\데이터 분석"
    r"\MULTICAM_11\프로젝트 전처리"
)

YOUTH_FILE = (
    BASE_DIR
    / "OD 데이터 전연령 (통합)"
    / "youth_mobility_seoul_cleaned.csv"
)

COMMUTE_DIR = (
    BASE_DIR
    / "OD 데이터 (이동수단 전처리 (출근))"
)

RETURN_DIR = (
    BASE_DIR
    / "OD 데이터 (이동수간 전처리 (귀가))"
)

OUTPUT_DIR = (
    BASE_DIR
    / "청년 이동패턴 검증"
)

MORNING_START = 7 * 60
MORNING_END = 9 * 60 + 40
AFTERNOON_START = 17 * 60
AFTERNOON_END = 19 * 60 + 40

## 2. 입력 파일 확인

In [2]:
print("청년 이동자료:", YOUTH_FILE.exists())

for month in range(1, 13):
    commute_file = (
        COMMUTE_DIR
        / f"{month}월 출근 목적 이동 데이터.csv"
    )

    return_file = (
        RETURN_DIR
        / f"{month}월 귀가 목적 이동 데이터.csv"
    )

    print(
        f"{month}월:",
        "출근 있음" if commute_file.exists() else "출근 없음",
        "/",
        "귀가 있음" if return_file.exists() else "귀가 없음"
    )

청년 이동자료: True
1월: 출근 있음 / 귀가 있음
2월: 출근 있음 / 귀가 있음
3월: 출근 있음 / 귀가 있음
4월: 출근 있음 / 귀가 있음
5월: 출근 있음 / 귀가 있음
6월: 출근 있음 / 귀가 있음
7월: 출근 있음 / 귀가 있음
8월: 출근 있음 / 귀가 있음
9월: 출근 있음 / 귀가 있음
10월: 출근 있음 / 귀가 있음
11월: 출근 있음 / 귀가 있음
12월: 출근 있음 / 귀가 있음


## 3. PDF 5번에 명시된 청년 데이터 품질 확인

다음 항목만 확인합니다.

- 이동시간이 0이거나 비정상적으로 긴 이동
- 이동거리가 0이거나 지나치게 긴 이동
- 이동시간 대비 이동거리가 비현실적인 이동
- 특정 OD 또는 시간대에 이동량이 과도하게 집중된 경우
- 동일한 출발동·도착동·시간대 조합의 중복
- 출발동과 도착동 코드의 누락 또는 유효성

임의의 수치 기준으로 이상치를 판정하거나 자동 삭제하지 않습니다. 분포와 상·하위 후보를 출력하여 직접 확인합니다.

In [3]:
def keep_extreme_rows(
    saved_rows,
    new_rows,
    column,
    largest=True,
    row_count=20
):
    combined = pd.concat(
        [saved_rows, new_rows],
        ignore_index=True
    )

    if largest:
        return combined.nlargest(row_count, column)

    return combined.nsmallest(row_count, column)


def check_youth_data():
    total_rows = 0
    duplicate_candidates = 0
    invalid_origin_codes = 0
    invalid_destination_codes = 0
    zero_time_count = 0
    zero_distance_count = 0
    missing_count = None

    numeric_columns = [
        "move_time",
        "move_dist",
        "2030_cnt"
    ]

    numeric_stats = {
        column: {
            "count": 0,
            "sum": 0.0,
            "min": np.inf,
            "max": -np.inf
        }
        for column in numeric_columns
    }

    top_rows = {
        column: pd.DataFrame()
        for column in numeric_columns
    }

    bottom_rows = {
        column: pd.DataFrame()
        for column in numeric_columns
    }

    relation_top = pd.DataFrame()
    od_amount = None
    time_amount = None

    for chunk in pd.read_csv(
        YOUTH_FILE,
        chunksize=200000,
        low_memory=False,
        encoding="utf-8-sig",
        dtype={
            "o_admdong_cd": "string",
            "d_admdong_cd": "string"
        }
    ):
        total_rows += len(chunk)

        if missing_count is None:
            missing_count = pd.Series(
                0,
                index=chunk.columns,
                dtype="int64"
            )

        missing_count = missing_count.add(
            chunk.isna().sum(),
            fill_value=0
        )

        duplicate_columns = [
            "o_admdong_cd",
            "d_admdong_cd",
            "st_time_cd"
        ]

        duplicate_candidates += chunk.duplicated(
            subset=duplicate_columns
        ).sum()

        origin_code = (
            chunk["o_admdong_cd"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
        )

        destination_code = (
            chunk["d_admdong_cd"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
        )

        invalid_origin_codes += (
            ~origin_code.str.fullmatch(
                r"[0-9]{8}",
                na=False
            )
        ).sum()

        invalid_destination_codes += (
            ~destination_code.str.fullmatch(
                r"[0-9]{8}",
                na=False
            )
        ).sum()

        numeric_data = {}

        for column in numeric_columns:
            values = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

            numeric_data[column] = values
            valid_values = values.dropna()

            if not valid_values.empty:
                numeric_stats[column]["count"] += len(valid_values)
                numeric_stats[column]["sum"] += valid_values.sum()
                numeric_stats[column]["min"] = min(
                    numeric_stats[column]["min"],
                    valid_values.min()
                )
                numeric_stats[column]["max"] = max(
                    numeric_stats[column]["max"],
                    valid_values.max()
                )

                value_rows = chunk[
                    [
                        "etl_ymd",
                        "o_admdong_cd",
                        "d_admdong_cd",
                        "st_time_cd",
                        column
                    ]
                ].copy()

                value_rows[column] = values
                value_rows = value_rows.dropna(subset=[column])

                top_rows[column] = keep_extreme_rows(
                    top_rows[column],
                    value_rows,
                    column,
                    largest=True
                )

                bottom_rows[column] = keep_extreme_rows(
                    bottom_rows[column],
                    value_rows,
                    column,
                    largest=False
                )

        zero_time_count += (
            numeric_data["move_time"] == 0
        ).sum()

        zero_distance_count += (
            numeric_data["move_dist"] == 0
        ).sum()

        valid_relation = (
            (numeric_data["move_time"] > 0)
            & numeric_data["move_dist"].notna()
        )

        relation_rows = chunk.loc[
            valid_relation,
            [
                "etl_ymd",
                "o_admdong_cd",
                "d_admdong_cd",
                "move_time",
                "move_dist"
            ]
        ].copy()

        relation_rows[
            "이동시간 대비 이동거리(km/h)"
        ] = (
            numeric_data["move_dist"].loc[valid_relation]
            / 1000
            / (
                numeric_data["move_time"].loc[valid_relation]
                / 60
            )
        )

        relation_top = keep_extreme_rows(
            relation_top,
            relation_rows,
            "이동시간 대비 이동거리(km/h)",
            largest=True
        )

        count_values = numeric_data["2030_cnt"].fillna(0)

        chunk_for_sum = chunk.assign(
            청년이동량=count_values
        )

        chunk_od_amount = chunk_for_sum.groupby(
            [
                "o_admdong_cd",
                "d_admdong_cd"
            ]
        )["청년이동량"].sum()

        if od_amount is None:
            od_amount = chunk_od_amount
        else:
            od_amount = od_amount.add(
                chunk_od_amount,
                fill_value=0
            )

        chunk_time_amount = chunk_for_sum.groupby(
            "st_time_cd"
        )["청년이동량"].sum()

        if time_amount is None:
            time_amount = chunk_time_amount
        else:
            time_amount = time_amount.add(
                chunk_time_amount,
                fill_value=0
            )

    basic_result = pd.DataFrame({
        "확인 항목": [
            "전체 행",
            "이동시간 0인 행",
            "이동거리 0인 행",
            "출발·도착·시간대 중복 후보",
            "출발 행정동 코드 누락·오류",
            "도착 행정동 코드 누락·오류"
        ],
        "개수": [
            total_rows,
            zero_time_count,
            zero_distance_count,
            duplicate_candidates,
            invalid_origin_codes,
            invalid_destination_codes
        ]
    })

    print("[기본 품질 확인]")
    display(basic_result)

    missing_result = pd.DataFrame({
        "결측치 개수": missing_count.astype("int64"),
        "결측치 비율(%)": (
            missing_count / total_rows * 100
        ).round(4)
    })

    missing_result = missing_result[
        missing_result["결측치 개수"] > 0
    ]

    print("\n[결측치 확인]")
    if missing_result.empty:
        print("결측치가 없습니다.")
    else:
        display(missing_result)

    distribution_rows = []

    for column in numeric_columns:
        stats = numeric_stats[column]

        mean_value = (
            stats["sum"] / stats["count"]
            if stats["count"] > 0
            else np.nan
        )

        distribution_rows.append({
            "컬럼": column,
            "유효값 개수": stats["count"],
            "평균": mean_value,
            "최솟값": stats["min"],
            "최댓값": stats["max"]
        })

    print("\n[이동시간·이동거리·이동량 분포]")
    display(pd.DataFrame(distribution_rows))

    for column in numeric_columns:
        print(f"\n[{column} 하위 20개]")
        display(bottom_rows[column])

        print(f"\n[{column} 상위 20개]")
        display(top_rows[column])

    print("\n[이동시간 대비 이동거리 상위 후보]")
    display(relation_top)

    print("\n[OD별 이동량 집중 상위]")
    display(
        od_amount
        .sort_values(ascending=False)
        .head(20)
        .rename("청년 이동량")
        .reset_index()
    )

    print("\n[시간대별 이동량 집중]")
    display(
        time_amount
        .sort_values(ascending=False)
        .rename("청년 이동량")
        .reset_index()
    )

    return basic_result

In [4]:
# 품질 확인은 시간이 걸리므로 필요할 때 True로 변경합니다.
RUN_QUALITY_CHECK = True

if RUN_QUALITY_CHECK:
    quality_result = check_youth_data()
else:
    print("현재는 품질 확인 코드를 실행하지 않습니다.")

[기본 품질 확인]


,확인 항목,개수
0,전체 행,217856493
1,이동시간 0인 행,4504356
2,이동거리 0인 행,496
3,출발·도착·시간대 중복 후보,77593807
4,출발 행정동 코드 누락·오류,0
5,도착 행정동 코드 누락·오류,0



[결측치 확인]
결측치가 없습니다.

[이동시간·이동거리·이동량 분포]


,컬럼,유효값 개수,평균,최솟값,최댓값
0,move_time,217856493,41.398038,0.0,779.0
1,move_dist,217856493,6361.540334,0.0,35100.0
2,2030_cnt,217856493,2.475886,0.0,1347.5



[move_time 하위 20개]


...

,etl_ymd,o_admdong_cd,d_admdong_cd,move_time,move_dist,이동시간 대비 이동거리(km/h)
0,2025-10-10,11740515,11500620,1,32651,1959.06
1,2025-06-30,11710540,11500620,1,32166,1929.96
2,2025-04-17,11740526,11500620,1,32137,1928.22
3,2025-04-21,11740526,11500620,1,32137,1928.22
4,2025-02-11,11740526,11500620,1,32072,1924.32
5,2025-06-20,11740526,11500620,1,32072,1924.32
6,2025-07-15,11740526,11500620,1,32072,1924.32
7,2025-09-16,11740526,11500620,1,32072,1924.32
8,2025-12-12,11740525,11500620,1,32028,1921.68
9,2025-03-12,11500620,11740526,1,31961,1917.66



[OD별 이동량 집중 상위]


,o_admdong_cd,d_admdong_cd,청년 이동량
0,11560540,11560540,2268158.92
1,11410585,11410585,2236054.22
2,11680640,11680640,2146518.64
3,11545510,11545510,1492942.47
4,11110615,11110615,1098187.41
5,11170625,11170625,1059015.71
6,11440660,11440660,989444.07
7,11650530,11650530,869865.53
8,11290600,11290600,839025.61
9,11530540,11530540,825778.21



[시간대별 이동량 집중]


,st_time_cd,청년 이동량
0,18:00,63926875.44
1,08:00,43131314.68
2,17:40,39051906.73
3,07:40,38658053.21
4,17:20,37970779.81
5,18:20,37820818.50
6,08:20,37542794.96
7,17:00,36485789.61
8,07:20,31974048.70
9,18:40,29590025.37


## 4. 오후 이동 방향 변환

오전 이동은 원래 방향을 사용합니다.

오후 이동과 전체연령 귀가 이동은 출근 방향과 비교하기 위해 출발동과 도착동을 서로 바꿉니다.

In [5]:
def reverse_od_direction(data):
    old_origin_code = data[
        "출발 행정동 코드"
    ].copy()

    old_origin_name = data[
        "출발 행정동"
    ].copy()

    data["출발 행정동 코드"] = data[
        "도착 행정동 코드"
    ]

    data["출발 행정동"] = data[
        "도착 행정동"
    ]

    data["도착 행정동 코드"] = old_origin_code
    data["도착 행정동"] = old_origin_name

    return data

## 5. OD 집계 결과 정리

In [6]:
def finish_od_aggregation(od_parts):
    all_od = pd.concat(
        od_parts,
        ignore_index=True
    )

    group_columns = [
        "출발 행정동 코드",
        "출발 행정동",
        "도착 행정동 코드",
        "도착 행정동"
    ]

    result = all_od.groupby(
        group_columns,
        as_index=False
    )[
        [
            "이동량",
            "이동시간 가중합",
            "이동거리 가중합"
        ]
    ].sum()

    result["평균 이동시간"] = (
        result["이동시간 가중합"]
        / result["이동량"]
    )

    result["평균 이동거리"] = (
        result["이동거리 가중합"]
        / result["이동량"]
    )

    return result[
        group_columns
        + [
            "이동량",
            "평균 이동시간",
            "평균 이동거리"
        ]
    ]

## 6. 청년 오전·오후 OD 집계

- 연령: 20~39세
- 요일: 평일
- 오전: 07:00~09:40, 원래 방향
- 오후: 17:00~19:40, 방향 반전

In [7]:
def aggregate_youth_od(period):
    od_parts = []

    for chunk in pd.read_csv(
        YOUTH_FILE,
        chunksize=200000,
        low_memory=False,
        encoding="utf-8-sig",
        dtype={
            "o_admdong_cd": "string",
            "d_admdong_cd": "string"
        }
    ):
        date_value = pd.to_datetime(
            chunk["etl_ymd"].astype("string"),
            format="%Y-%m-%d",
            errors="coerce"
        )

        weekday = date_value.dt.dayofweek < 5

        start_time = pd.to_datetime(
            chunk["st_time_cd"],
            format="%H:%M",
            errors="coerce"
        )

        start_minutes = (
            start_time.dt.hour * 60
            + start_time.dt.minute
        )

        if period == "오전":
            time_condition = start_minutes.between(
                MORNING_START,
                MORNING_END
            )
        elif period == "오후":
            time_condition = start_minutes.between(
                AFTERNOON_START,
                AFTERNOON_END
            )
        else:
            raise ValueError(
                "period는 '오전' 또는 '오후'만 입력합니다."
            )

        selected = chunk[
            weekday & time_condition
        ].copy()

        if selected.empty:
            continue

        selected = selected.rename(
            columns={
                "o_admdong_cd": "출발 행정동 코드",
                "o_admdong_name": "출발 행정동",
                "d_admdong_cd": "도착 행정동 코드",
                "d_admdong_name": "도착 행정동",
                "2030_cnt": "이동량",
                "move_time": "평균 이동시간",
                "move_dist": "평균 이동거리"
            }
        )

        for column in [
            "이동량",
            "평균 이동시간",
            "평균 이동거리"
        ]:
            selected[column] = pd.to_numeric(
                selected[column],
                errors="coerce"
            )

        if period == "오후":
            selected = reverse_od_direction(selected)

        selected["이동시간 가중합"] = (
            selected["이동량"]
            * selected["평균 이동시간"]
        )

        selected["이동거리 가중합"] = (
            selected["이동량"]
            * selected["평균 이동거리"]
        )

        group_columns = [
            "출발 행정동 코드",
            "출발 행정동",
            "도착 행정동 코드",
            "도착 행정동"
        ]

        part = selected.groupby(
            group_columns,
            as_index=False
        )[
            [
                "이동량",
                "이동시간 가중합",
                "이동거리 가중합"
            ]
        ].sum()

        od_parts.append(part)

    if not od_parts:
        raise ValueError(
            f"청년 {period} 조건에 맞는 데이터가 없습니다."
        )

    return finish_od_aggregation(od_parts)

## 7. 전체연령 출근·귀가 OD 집계

- 출근 목적 코드: `1`
- 귀가 목적 코드: `3`
- 귀가 OD는 방향을 반대로 변환

In [8]:
def aggregate_admdong3_od(purpose_code, reverse=False):
    od_parts = []

    source_directory = (
        RETURN_DIR
        if reverse
        else COMMUTE_DIR
    )

    purpose_name = (
        "귀가"
        if reverse
        else "출근"
    )

    for month in range(1, 13):
        file_path = (
            source_directory
            / f"{month}월 {purpose_name} 목적 이동 데이터.csv"
        )

        if not file_path.exists():
            raise FileNotFoundError(
                f"파일을 찾지 못했습니다: {file_path}"
            )

        for chunk in pd.read_csv(
            file_path,
            chunksize=200000,
            low_memory=False,
            encoding="utf-8-sig",
            dtype={
                "출발 행정동 코드": "string",
                "도착 행정동 코드": "string"
            }
        ):
            move_purpose = pd.to_numeric(
                chunk["이동 목적 코드"],
                errors="coerce"
            )

            selected = chunk[
                move_purpose == purpose_code
            ].copy()

            if selected.empty:
                continue

            selected = selected.rename(
                columns={
                    "이동 인구수": "이동량",
                    "이동 시간": "평균 이동시간",
                    "이동거리": "평균 이동거리"
                }
            )

            for column in [
                "이동량",
                "평균 이동시간",
                "평균 이동거리"
            ]:
                selected[column] = pd.to_numeric(
                    selected[column],
                    errors="coerce"
                )

            if reverse:
                selected = reverse_od_direction(selected)

            selected["이동시간 가중합"] = (
                selected["이동량"]
                * selected["평균 이동시간"]
            )

            selected["이동거리 가중합"] = (
                selected["이동량"]
                * selected["평균 이동거리"]
            )

            group_columns = [
                "출발 행정동 코드",
                "출발 행정동",
                "도착 행정동 코드",
                "도착 행정동"
            ]

            part = selected.groupby(
                group_columns,
                as_index=False
            )[
                [
                    "이동량",
                    "이동시간 가중합",
                    "이동거리 가중합"
                ]
            ].sum()

            od_parts.append(part)

        print(f"{month}월 {purpose_name} 집계 완료")

    if not od_parts:
        raise ValueError(
            f"전체연령 {purpose_name} 데이터가 없습니다."
        )

    return finish_od_aggregation(od_parts)

## 8. 목적지 비중 계산

```text
목적지 비중
= 해당 OD 이동량
÷ 해당 출발동의 전체 이동량
```

In [9]:
def add_destination_share(data):
    result = data.copy()

    origin_total = result.groupby(
        "출발 행정동 코드"
    )["이동량"].transform("sum")

    result["목적지 비중"] = (
        result["이동량"]
        / origin_total
    )

    return result


def summarize_origin(data):
    result = data.copy()

    result["이동시간 가중합"] = (
        result["이동량"]
        * result["평균 이동시간"]
    )

    result["이동거리 가중합"] = (
        result["이동량"]
        * result["평균 이동거리"]
    )

    summary = result.groupby(
        [
            "출발 행정동 코드",
            "출발 행정동"
        ],
        as_index=False
    )[
        [
            "이동량",
            "이동시간 가중합",
            "이동거리 가중합"
        ]
    ].sum()

    summary["평균 이동시간"] = (
        summary["이동시간 가중합"]
        / summary["이동량"]
    )

    summary["평균 이동거리"] = (
        summary["이동거리 가중합"]
        / summary["이동량"]
    )

    return summary[
        [
            "출발 행정동 코드",
            "출발 행정동",
            "이동량",
            "평균 이동시간",
            "평균 이동거리"
        ]
    ]

## 9. 청년 이동패턴 검증의 네 가지 비교 지표

```text
상위 5개 목적지 일치율
= 공통으로 등장한 목적지 수 ÷ 5
```

```text
목적지 비중 코사인 유사도
= 두 목적지 비중 벡터의 내적
÷ 두 벡터 크기의 곱
```

```text
평균 이동시간 차이
= 청년 평균 이동시간 - 전체연령 평균 이동시간
```

```text
평균 이동거리 차이
= 청년 평균 이동거리 - 전체연령 평균 이동거리
```

In [10]:
def compare_destination_patterns(
    reference_od,
    youth_od,
    result_name,
    output_file
):
    reference = add_destination_share(reference_od)
    youth = add_destination_share(youth_od)

    reference_top5 = (
        reference
        .sort_values(
            [
                "출발 행정동 코드",
                "목적지 비중"
            ],
            ascending=[True, False]
        )
        .groupby("출발 행정동 코드")
        .head(5)
    )

    youth_top5 = (
        youth
        .sort_values(
            [
                "출발 행정동 코드",
                "목적지 비중"
            ],
            ascending=[True, False]
        )
        .groupby("출발 행정동 코드")
        .head(5)
    )

    common_top5 = reference_top5[
        [
            "출발 행정동 코드",
            "도착 행정동 코드"
        ]
    ].merge(
        youth_top5[
            [
                "출발 행정동 코드",
                "도착 행정동 코드"
            ]
        ],
        on=[
            "출발 행정동 코드",
            "도착 행정동 코드"
        ],
        how="inner"
    )

    top5_result = common_top5.groupby(
        "출발 행정동 코드",
        as_index=False
    ).size()

    top5_result = top5_result.rename(
        columns={
            "size": "상위 5개 공통 목적지 수"
        }
    )

    top5_result["상위 5개 목적지 일치율"] = (
        top5_result["상위 5개 공통 목적지 수"]
        / 5
    )

    share_compare = reference[
        [
            "출발 행정동 코드",
            "도착 행정동 코드",
            "목적지 비중"
        ]
    ].rename(
        columns={
            "목적지 비중": "전체연령 목적지 비중"
        }
    ).merge(
        youth[
            [
                "출발 행정동 코드",
                "도착 행정동 코드",
                "목적지 비중"
            ]
        ].rename(
            columns={
                "목적지 비중": "청년 목적지 비중"
            }
        ),
        on=[
            "출발 행정동 코드",
            "도착 행정동 코드"
        ],
        how="outer"
    ).fillna(0)

    cosine_rows = []

    for origin_code, group in share_compare.groupby(
        "출발 행정동 코드"
    ):
        reference_vector = group[
            "전체연령 목적지 비중"
        ]

        youth_vector = group[
            "청년 목적지 비중"
        ]

        dot_product = (
            reference_vector * youth_vector
        ).sum()

        denominator = (
            np.sqrt((reference_vector ** 2).sum())
            * np.sqrt((youth_vector ** 2).sum())
        )

        cosine_similarity = (
            dot_product / denominator
            if denominator > 0
            else np.nan
        )

        cosine_rows.append({
            "출발 행정동 코드": origin_code,
            "목적지 비중 코사인 유사도":
                cosine_similarity
        })

    cosine_result = pd.DataFrame(cosine_rows)

    reference_summary = summarize_origin(
        reference_od
    ).rename(
        columns={
            "출발 행정동": "출발 행정동",
            "이동량": "전체연령 이동량",
            "평균 이동시간": "전체연령 평균 이동시간",
            "평균 이동거리": "전체연령 평균 이동거리"
        }
    )

    youth_summary = summarize_origin(
        youth_od
    ).rename(
        columns={
            "이동량": "청년 이동량",
            "평균 이동시간": "청년 평균 이동시간",
            "평균 이동거리": "청년 평균 이동거리"
        }
    )

    result = reference_summary.merge(
        youth_summary[
            [
                "출발 행정동 코드",
                "청년 이동량",
                "청년 평균 이동시간",
                "청년 평균 이동거리"
            ]
        ],
        on="출발 행정동 코드",
        how="outer"
    )

    result = result.merge(
        top5_result,
        on="출발 행정동 코드",
        how="left"
    )

    result = result.merge(
        cosine_result,
        on="출발 행정동 코드",
        how="left"
    )

    result["상위 5개 공통 목적지 수"] = (
        result["상위 5개 공통 목적지 수"]
        .fillna(0)
        .astype("int64")
    )

    result["상위 5개 목적지 일치율"] = (
        result["상위 5개 목적지 일치율"]
        .fillna(0)
    )

    result["평균 이동시간 차이"] = (
        result["청년 평균 이동시간"]
        - result["전체연령 평균 이동시간"]
    )

    result["평균 이동거리 차이"] = (
        result["청년 평균 이동거리"]
        - result["전체연령 평균 이동거리"]
    )

    result.insert(0, "비교 구분", result_name)

    result.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("생성 파일:", output_file)

    return result

## 10. 전체 검증 실행

오전 결과, 오후 결과, 오전·오후를 나란히 결합한 종합 결과를 생성합니다.

In [11]:
def run_youth_pattern_validation():
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    print("[1] 전체연령 출근 OD 집계")
    all_age_commute = aggregate_admdong3_od(
        purpose_code=1,
        reverse=False
    )

    print("\n[2] 전체연령 귀가 OD 집계 및 방향 반전")
    all_age_return = aggregate_admdong3_od(
        purpose_code=3,
        reverse=True
    )

    print("\n[3] 청년 오전 OD 집계")
    youth_morning = aggregate_youth_od("오전")

    print("\n[4] 청년 오후 OD 집계 및 방향 반전")
    youth_afternoon = aggregate_youth_od("오후")

    morning_output = (
        OUTPUT_DIR
        / "2025년 청년 오전 출근패턴 검증.csv"
    )

    afternoon_output = (
        OUTPUT_DIR
        / "2025년 청년 오후 귀가패턴 검증.csv"
    )

    combined_output = (
        OUTPUT_DIR
        / "2025년 청년 이동패턴 종합 검증.csv"
    )

    morning_result = compare_destination_patterns(
        all_age_commute,
        youth_morning,
        "전체연령 출근 OD와 청년 오전 OD",
        morning_output
    )

    afternoon_result = compare_destination_patterns(
        all_age_return,
        youth_afternoon,
        "전체연령 귀가 OD와 청년 오후 OD",
        afternoon_output
    )

    morning_columns = {
        column: f"오전 {column}"
        for column in morning_result.columns
        if column not in [
            "출발 행정동 코드",
            "출발 행정동"
        ]
    }

    afternoon_columns = {
        column: f"오후 {column}"
        for column in afternoon_result.columns
        if column not in [
            "출발 행정동 코드",
            "출발 행정동"
        ]
    }

    combined_result = morning_result.rename(
        columns=morning_columns
    ).merge(
        afternoon_result.rename(
            columns=afternoon_columns
        ),
        on="출발 행정동 코드",
        how="outer",
        suffixes=(
            "_오전",
            "_오후"
        )
    )

    combined_result.to_csv(
        combined_output,
        index=False,
        encoding="utf-8-sig"
    )

    print("생성 파일:", combined_output)

    return (
        morning_result,
        afternoon_result,
        combined_result
    )

## 11. 실행

코드를 확인한 후 `RUN_PROCESSING`을 `True`로 변경하면 결과 파일 세 개를 생성합니다.

In [12]:
RUN_PROCESSING = True

if RUN_PROCESSING:
    (
        morning_result,
        afternoon_result,
        combined_result
    ) = run_youth_pattern_validation()
else:
    print("현재는 결과 CSV를 생성하지 않습니다.")

[1] 전체연령 출근 OD 집계
1월 출근 집계 완료
2월 출근 집계 완료
3월 출근 집계 완료
4월 출근 집계 완료
5월 출근 집계 완료
6월 출근 집계 완료
7월 출근 집계 완료
8월 출근 집계 완료
9월 출근 집계 완료
10월 출근 집계 완료
11월 출근 집계 완료
12월 출근 집계 완료

[2] 전체연령 귀가 OD 집계 및 방향 반전
1월 귀가 집계 완료
2월 귀가 집계 완료
3월 귀가 집계 완료
4월 귀가 집계 완료
5월 귀가 집계 완료
6월 귀가 집계 완료
7월 귀가 집계 완료
8월 귀가 집계 완료
9월 귀가 집계 완료
10월 귀가 집계 완료
11월 귀가 집계 완료
12월 귀가 집계 완료

[3] 청년 오전 OD 집계

[4] 청년 오후 OD 집계 및 방향 반전
생성 파일: C:\Users\Owner\OneDrive\바탕 화면\데이터 분석\MULTICAM_11\프로젝트 전처리\청년 이동패턴 검증\2025년 청년 오전 출근패턴 검증.csv
생성 파일: C:\Users\Owner\OneDrive\바탕 화면\데이터 분석\MULTICAM_11\프로젝트 전처리\청년 이동패턴 검증\2025년 청년 오후 귀가패턴 검증.csv
생성 파일: C:\Users\Owner\OneDrive\바탕 화면\데이터 분석\MULTICAM_11\프로젝트 전처리\청년 이동패턴 검증\2025년 청년 이동패턴 종합 검증.csv


## 12. 결과 확인

In [13]:
# 실행 후 필요할 때 주석을 해제합니다.

display(morning_result.head(10))
display(afternoon_result.head(10))
display(combined_result.head(10))

,비교 구분,출발 행정동 코드,출발 행정동,전체연령 이동량,전체연령 평균 이동시간,전체연령 평균 이동거리,청년 이동량,청년 평균 이동시간,청년 평균 이동거리,상위 5개 공통 목적지 수,상위 5개 목적지 일치율,목적지 비중 코사인 유사도,평균 이동시간 차이,평균 이동거리 차이
0,전체연령 출근 OD와 청년 오전 OD,11110515,청운효자동,850604.05,34.739946,4641.031032,273490.24,29.743193,4696.609086,5,1.0,0.972477,-4.996753,55.578054
1,전체연령 출근 OD와 청년 오전 OD,11110530,사직동,1276783.97,28.888731,3295.410051,365346.18,24.512944,3584.836480,5,1.0,0.986256,-4.375787,289.426429
2,전체연령 출근 OD와 청년 오전 OD,11110540,삼청동,225791.18,34.389623,4044.355542,63162.20,28.668301,4209.045214,5,1.0,0.943208,-5.721322,164.689672
3,전체연령 출근 OD와 청년 오전 OD,11110550,부암동,674176.45,39.351804,5657.979740,199265.77,36.986368,6142.918308,5,1.0,0.977872,-2.365436,484.938568
4,전체연령 출근 OD와 청년 오전 OD,11110560,평창동,1028915.34,41.741869,6194.700315,278850.66,40.724441,6692.944397,5,1.0,0.979610,-1.017428,498.244083
5,전체연령 출근 OD와 청년 오전 OD,11110570,무악동,317153.81,34.479110,4617.020250,94132.58,29.272869,4790.310812,4,0.8,0.967018,-5.206241,173.290562
6,전체연령 출근 OD와 청년 오전 OD,11110580,교남동,880562.78,34.460491,4244.500595,287105.70,28.739076,4264.562094,5,1.0,0.987597,-5.721415,20.061499
7,전체연령 출근 OD와 청년 오전 OD,11110600,가회동,329463.37,33.318570,3970.519808,91492.02,28.893493,4430.297314,4,0.8,0.976935,-4.425077,459.777506
8,전체연령 출근 OD와 청년 오전 OD,11110615,종로1.2.3.4가동,2186428.61,29.093358,3075.728252,568654.23,22.024279,2830.108652,3,0.6,0.986437,-7.069079,-245.619600
9,전체연령 출근 OD와 청년 오전 OD,11110630,종로5.6가동,820886.58,31.821086,3390.259009,259442.32,22.932459,3012.009974,4,0.8,0.842442,-8.888626,-378.249035


,비교 구분,출발 행정동 코드,출발 행정동,전체연령 이동량,전체연령 평균 이동시간,전체연령 평균 이동거리,청년 이동량,청년 평균 이동시간,청년 평균 이동거리,상위 5개 공통 목적지 수,상위 5개 목적지 일치율,목적지 비중 코사인 유사도,평균 이동시간 차이,평균 이동거리 차이
0,전체연령 귀가 OD와 청년 오후 OD,11110515,청운효자동,1914882.02,35.303331,3558.686710,401398.84,30.921090,3782.516034,5,1.0,0.981971,-4.382241,223.829325
1,전체연령 귀가 OD와 청년 오후 OD,11110530,사직동,1915822.21,30.252573,2873.243320,1103136.94,28.770048,3557.246101,5,1.0,0.877089,-1.482525,684.002780
2,전체연령 귀가 OD와 청년 오후 OD,11110540,삼청동,396563.16,34.740098,3152.272126,169980.39,28.915979,3249.568499,5,1.0,0.919282,-5.824119,97.296373
3,전체연령 귀가 OD와 청년 오후 OD,11110550,부암동,1692785.15,37.250900,4356.538471,186259.32,33.696252,4844.378533,5,1.0,0.992443,-3.554648,487.840062
4,전체연령 귀가 OD와 청년 오후 OD,11110560,평창동,2666313.92,37.954818,4855.465638,234409.52,36.731964,5576.884638,5,1.0,0.981347,-1.222854,721.419000
5,전체연령 귀가 OD와 청년 오후 OD,11110570,무악동,719329.04,32.861099,3650.191016,90786.83,30.803248,4346.949280,4,0.8,0.925786,-2.057851,696.758264
6,전체연령 귀가 OD와 청년 오후 OD,11110580,교남동,1816033.82,33.277243,3228.815569,263572.01,28.000835,3542.112929,4,0.8,0.924672,-5.276408,313.297361
7,전체연령 귀가 OD와 청년 오후 OD,11110600,가회동,599566.37,33.595952,3112.742091,239424.95,30.787589,3686.003690,4,0.8,0.923317,-2.808363,573.261598
8,전체연령 귀가 OD와 청년 오후 OD,11110615,종로1.2.3.4가동,2135814.43,28.868916,2494.878918,3664913.17,27.427037,3518.881738,4,0.8,0.970908,-1.441879,1024.002820
9,전체연령 귀가 OD와 청년 오후 OD,11110630,종로5.6가동,984812.58,30.107800,2408.342129,536747.57,26.651692,3387.996046,5,1.0,0.961846,-3.456108,979.653916


,오전 비교 구분,출발 행정동 코드,출발 행정동_오전,오전 전체연령 이동량,오전 전체연령 평균 이동시간,오전 전체연령 평균 이동거리,오전 청년 이동량,오전 청년 평균 이동시간,오전 청년 평균 이동거리,오전 상위 5개 공통 목적지 수,...,오후 전체연령 평균 이동시간,오후 전체연령 평균 이동거리,오후 청년 이동량,오후 청년 평균 이동시간,오후 청년 평균 이동거리,오후 상위 5개 공통 목적지 수,오후 상위 5개 목적지 일치율,오후 목적지 비중 코사인 유사도,오후 평균 이동시간 차이,오후 평균 이동거리 차이
0,전체연령 출근 OD와 청년 오전 OD,11110515,청운효자동,850604.05,34.739946,4641.031032,273490.24,29.743193,4696.609086,5,...,35.303331,3558.686710,401398.84,30.921090,3782.516034,5,1.0,0.981971,-4.382241,223.829325
1,전체연령 출근 OD와 청년 오전 OD,11110530,사직동,1276783.97,28.888731,3295.410051,365346.18,24.512944,3584.836480,5,...,30.252573,2873.243320,1103136.94,28.770048,3557.246101,5,1.0,0.877089,-1.482525,684.002780
2,전체연령 출근 OD와 청년 오전 OD,11110540,삼청동,225791.18,34.389623,4044.355542,63162.20,28.668301,4209.045214,5,...,34.740098,3152.272126,169980.39,28.915979,3249.568499,5,1.0,0.919282,-5.824119,97.296373
3,전체연령 출근 OD와 청년 오전 OD,11110550,부암동,674176.45,39.351804,5657.979740,199265.77,36.986368,6142.918308,5,...,37.250900,4356.538471,186259.32,33.696252,4844.378533,5,1.0,0.992443,-3.554648,487.840062
4,전체연령 출근 OD와 청년 오전 OD,11110560,평창동,1028915.34,41.741869,6194.700315,278850.66,40.724441,6692.944397,5,...,37.954818,4855.465638,234409.52,36.731964,5576.884638,5,1.0,0.981347,-1.222854,721.419000
5,전체연령 출근 OD와 청년 오전 OD,11110570,무악동,317153.81,34.479110,4617.020250,94132.58,29.272869,4790.310812,4,...,32.861099,3650.191016,90786.83,30.803248,4346.949280,4,0.8,0.925786,-2.057851,696.758264
6,전체연령 출근 OD와 청년 오전 OD,11110580,교남동,880562.78,34.460491,4244.500595,287105.70,28.739076,4264.562094,5,...,33.277243,3228.815569,263572.01,28.000835,3542.112929,4,0.8,0.924672,-5.276408,313.297361
7,전체연령 출근 OD와 청년 오전 OD,11110600,가회동,329463.37,33.318570,3970.519808,91492.02,28.893493,4430.297314,4,...,33.595952,3112.742091,239424.95,30.787589,3686.003690,4,0.8,0.923317,-2.808363,573.261598
8,전체연령 출근 OD와 청년 오전 OD,11110615,종로1.2.3.4가동,2186428.61,29.093358,3075.728252,568654.23,22.024279,2830.108652,3,...,28.868916,2494.878918,3664913.17,27.427037,3518.881738,4,0.8,0.970908,-1.441879,1024.002820
9,전체연령 출근 OD와 청년 오전 OD,11110630,종로5.6가동,820886.58,31.821086,3390.259009,259442.32,22.932459,3012.009974,4,...,30.107800,2408.342129,536747.57,26.651692,3387.996046,5,1.0,0.961846,-3.456108,979.653916


## 13. 해석 시 주의사항

- 청년 자료에는 이동목적 변수가 없으므로 오전·오후 시간대 이동을 정확한 출퇴근으로 단정하지 않습니다.
- 네 가지 값은 전체연령 통근구조를 청년층에 적용할 수 있는지 검증하는 값입니다.
- 전체연령과 청년 이동량을 서로 합산하지 않습니다.
- 품질 확인에서 발견된 후보는 원자료 오류인지 실제 이동인지 확인하기 전까지 자동으로 삭제하지 않습니다.